# CS221 ViHOS — Stage D: PhoBERT + Linear-Chain CRF Layer (Google Colab GPU)

**Đề tài:** Phát hiện đoạn ngôn từ thù ghét và xúc phạm trong tiếng Việt (ViHOS Spans Detection)

**Động lực khoa học (Từ Giả thuyết H3):**
- Ở Run 02 (PhoBERT) và Run 03 (ViSoBERT), chúng ta phát hiện **75%–76% lỗi của mô hình là do sai lệch ranh giới (Over-span & Under-span)**.
- Nguyên nhân: Các mô hình Transformer thông thường dùng Softmax độc lập cho từng token, dẫn đến hiện tượng Self-Attention lan tỏa ngữ cảnh sang các từ lân cận (gây Over-span).
- **Giải pháp:** Tích hợp tầng **Linear-Chain CRF** vào đỉnh của PhoBERT. Tầng CRF sẽ học ma trận xác suất chuyển tiếp giữa các nhãn BIO ($P(B-T \rightarrow I-T)$ và cấm các chuyển tiếp vô lý), giúp **kẹp chặt ranh giới span**!

## 1. Cài đặt thư viện & Kiểm tra GPU

In [ ]:
!pip install -q transformers datasets accelerate seqeval scikit-learn pandas

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("CẢNH BÁO: Hãy bật GPU trong Runtime -> Change runtime type -> T4 GPU")

## 2. Nạp dữ liệu ViHOS chính thức

In [ ]:
import os
if not os.path.exists("DATA/ViHOS_official"):
    !git clone https://huggingface.co/datasets/phusroyal/ViHOS.git DATA/ViHOS_official
    print("Đã tải thành công ViHOS official!")
else:
    print("Dữ liệu đã có sẵn trong DATA/ViHOS_official")

## 3. Cấu hình Thử nghiệm

In [ ]:
MODEL_NAME = "vinai/phobert-base-v2"
MODE = "syllable"
BATCH_SIZE = 16
LEARNING_RATE_TRANSFORMER = 2e-5
LEARNING_RATE_CRF = 1e-3          # Tầng CRF cần learning rate cao hơn một chút
NUM_EPOCHS = 3
MAX_LENGTH = 160
SEED = 42

LABEL2ID = {"O": 0, "B-T": 1, "I-T": 2}
ID2LABEL = {0: "O", 1: "B-T", 2: "I-T"}

print(f"Cấu hình Kiến trúc: {MODEL_NAME} + Linear-Chain CRF | Mode={MODE}")

## 4. Định nghĩa Kiến trúc Mô hình Lai: Transformer + Linear-Chain CRF

In [ ]:
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from typing import List, Optional, Tuple

class PyTorchCRF(nn.Module):
    def __init__(self, num_tags: int = 3):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        numerator = self._compute_score(emissions, tags, mask)
        denominator = self._compute_normalizer(emissions, mask)
        return -(numerator - denominator).mean()

    def decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        return self._viterbi_decode(emissions, mask)

    def _compute_score(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        seq_len, batch_size, _ = emissions.shape
        score = self.start_transitions[tags[0]] + emissions[0, torch.arange(batch_size), tags[0]]
        for i in range(1, seq_len):
            mask_i = mask[i]
            trans_score = self.transitions[tags[i], tags[i - 1]]
            emission_score = emissions[i, torch.arange(batch_size), tags[i]]
            score = score + (trans_score + emission_score) * mask_i
        last_valid = mask.sum(dim=0).long() - 1
        last_tags = tags[last_valid, torch.arange(batch_size)]
        score = score + self.end_transitions[last_tags]
        return score

    def _compute_normalizer(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        seq_len, batch_size, _ = emissions.shape
        forward_var = self.start_transitions + emissions[0]
        for i in range(1, seq_len):
            broadcast_emit = emissions[i].unsqueeze(2)
            broadcast_trans = self.transitions.unsqueeze(0)
            broadcast_var = forward_var.unsqueeze(1)
            next_var = torch.logsumexp(broadcast_var + broadcast_trans + broadcast_emit, dim=2)
            forward_var = torch.where(mask[i].unsqueeze(1).bool(), next_var, forward_var)
        forward_var = forward_var + self.end_transitions
        return torch.logsumexp(forward_var, dim=1)

    def _viterbi_decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        seq_len, batch_size, _ = emissions.shape
        viterbi_vars = self.start_transitions + emissions[0]
        backpointers = []
        for i in range(1, seq_len):
            broadcast_var = viterbi_vars.unsqueeze(1)
            broadcast_trans = self.transitions.unsqueeze(0)
            max_vars, bptrs = torch.max(broadcast_var + broadcast_trans, dim=2)
            max_vars = max_vars + emissions[i]
            viterbi_vars = torch.where(mask[i].unsqueeze(1).bool(), max_vars, viterbi_vars)
            backpointers.append(bptrs)
        viterbi_vars = viterbi_vars + self.end_transitions
        best_tags_list = []
        last_valid = mask.sum(dim=0).long() - 1
        for b in range(batch_size):
            last_i = last_valid[b].item()
            best_tag = torch.argmax(viterbi_vars[b]).item()
            best_tags = [best_tag]
            for i in range(last_i - 1, -1, -1):
                best_tag = backpointers[i][b, best_tag].item()
                best_tags.append(best_tag)
            best_tags.reverse()
            best_tags_list.append(best_tags)
        return best_tags_list


class PhoBERT_CRF(nn.Module):
    def __init__(self, model_name: str = "vinai/phobert-base-v2", num_tags: int = 3, dropout_rate: float = 0.2):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size, num_tags)
        self.crf = PyTorchCRF(num_tags)

    def forward(self, input_ids, attention_mask, first_subword_indices, word_labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        batch_size = sequence_output.size(0)
        device = sequence_output.device
        
        max_words = max(len(idxs) for idxs in first_subword_indices)
        word_emissions = torch.zeros(batch_size, max_words, sequence_output.size(-1), device=device)
        word_mask = torch.zeros(batch_size, max_words, dtype=torch.uint8, device=device)
        
        for b in range(batch_size):
            idxs = first_subword_indices[b]
            num_w = len(idxs)
            if num_w > 0:
                valid_idxs = [min(idx, sequence_output.size(1) - 1) for idx in idxs]
                word_emissions[b, :num_w] = sequence_output[b, valid_idxs]
                word_mask[b, :num_w] = 1
                
        emissions = self.classifier(word_emissions)
        emissions_crf = emissions.transpose(0, 1)
        word_mask_crf = word_mask.transpose(0, 1)
        
        loss = None
        if word_labels is not None:
            padded_labels = torch.zeros(batch_size, max_words, dtype=torch.long, device=device)
            for b in range(batch_size):
                labels_b = word_labels[b]
                num_w = min(len(labels_b), max_words)
                padded_labels[b, :num_w] = torch.tensor(labels_b[:num_w], dtype=torch.long, device=device)
            padded_labels_crf = padded_labels.transpose(0, 1)
            loss = self.crf(emissions_crf, padded_labels_crf, word_mask_crf)
            
        decoded_paths = self.crf.decode(emissions_crf, word_mask_crf)
        return loss, decoded_paths

print("Đã xây dựng thành công kiến trúc PhoBERT + CRF!")

## 5. Chuẩn bị DataLoader & Tokenization

In [ ]:
import ast
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class ViHOSSample:
    def __init__(self, sentence_id: int, content: str, tokens: List[str], tags: List[str], gold_char_offsets: set):
        self.sentence_id = sentence_id
        self.content = content
        self.tokens = tokens
        self.tags = tags
        self.gold_char_offsets = gold_char_offsets
        self.is_toxic = 1 if ("B-T" in tags or len(gold_char_offsets) > 0) else 0

def align_tokens_to_char_spans(content: str, tokens: List[str]):
    token_spans, curr_idx, content_lower = [], 0, content.lower()
    for token in tokens:
        tok_clean = str(token).strip()
        idx = content.find(tok_clean, curr_idx)
        if idx == -1: idx = content_lower.find(tok_clean.lower(), curr_idx)
        if idx != -1:
            start, end = idx, idx + len(tok_clean)
            token_spans.append((start, end))
            curr_idx = end
        else:
            start, end = curr_idx, min(curr_idx + len(tok_clean), len(content))
            token_spans.append((start, end))
            curr_idx = end
    return token_spans

def bio_to_char_offsets(tokens: List[str], tags: List[str], content: str):
    token_spans = align_tokens_to_char_spans(content, tokens)
    char_offsets, prev_end, prev_tag = set(), None, None
    for (start, end), tag in zip(token_spans, tags):
        if tag in ("B-T", "I-T"):
            if tag == "I-T" and prev_tag in ("B-T", "I-T") and prev_end is not None and prev_end < start:
                for c in range(prev_end, start): char_offsets.add(c)
            for c in range(start, end): char_offsets.add(c)
        prev_end, prev_tag = end, tag
    return char_offsets

def load_vihos_split(split: str = "train", mode: str = "syllable", data_dir: str = "DATA/ViHOS_official"):
    bio_file = f"{data_dir}/train_sequence_labeling/{mode}/{split}_BIO_syllable.csv" if mode == "syllable" else f"{data_dir}/train_sequence_labeling/{mode}/{split}_BIO_Word.csv"
    span_file = f"{data_dir}/train_span_extraction/{split}.csv" if split in ("train", "dev") else f"{data_dir}/test/test.csv"
    df_span, df_bio = pd.read_csv(span_file), pd.read_csv(bio_file)
    grouped_bio = df_bio.groupby("sentence_id")
    samples = []
    for sid in range(len(df_span)):
        span_row = df_span.iloc[sid]
        content = str(span_row["content"]) if pd.notna(span_row["content"]) else ""
        raw_spans = span_row.get("index_spans", "[]")
        gold_char_offsets = set(ast.literal_eval(str(raw_spans))) if pd.notna(raw_spans) and str(raw_spans).strip() not in ("", "nan") else set()
        if sid in grouped_bio.groups:
            grp = grouped_bio.get_group(sid)
            tokens = [str(w) if pd.notna(w) else "" for w in grp["Word"].tolist()]
            tags = [str(t) if pd.notna(t) else "O" for t in grp["Tag"].tolist()]
        else:
            tokens = content.split()
            tags = ["O"] * len(tokens)
        samples.append(ViHOSSample(sid, content, tokens, tags, gold_char_offsets))
    return samples

train_samples = load_vihos_split("train", mode=MODE)
dev_samples = load_vihos_split("dev", mode=MODE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ViHOSDataset(Dataset):
    def __init__(self, samples: List[ViHOSSample], tokenizer, max_length: int = 160):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = [self._process(s) for s in samples]

    def _process(self, sample):
        tokens, tags = sample.tokens, sample.tags
        bos = self.tokenizer.bos_token or self.tokenizer.cls_token
        eos = self.tokenizer.eos_token or self.tokenizer.sep_token
        subwords = [bos]
        first_subword_indices = []
        for word in tokens:
            w_tokens = self.tokenizer.tokenize(str(word))
            if not w_tokens: w_tokens = [self.tokenizer.unk_token]
            first_subword_indices.append(len(subwords))
            subwords.extend(w_tokens)
        subwords.append(eos)
        if len(subwords) > self.max_length:
            subwords = subwords[:self.max_length - 1] + [eos]
            first_subword_indices = [idx for idx in first_subword_indices if idx < self.max_length - 1]
        input_ids = self.tokenizer.convert_tokens_to_ids(subwords)
        pad_len = self.max_length - len(input_ids)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 1
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids += [pad_id] * pad_len
        word_label_ids = [LABEL2ID.get(t, 0) for t in tags[:len(first_subword_indices)]]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "first_subword_indices": first_subword_indices,
            "word_labels": word_label_ids,
            "sentence_id": sample.sentence_id
        }
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    first_subword_indices = [b["first_subword_indices"] for b in batch]
    word_labels = [b["word_labels"] for b in batch]
    sentence_ids = [b["sentence_id"] for b in batch]
    return input_ids, attention_mask, first_subword_indices, word_labels, sentence_ids

train_loader = DataLoader(ViHOSDataset(train_samples, tokenizer, max_length=MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(ViHOSDataset(dev_samples, tokenizer, max_length=MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"DataLoader đã sẵn sàng: Train = {len(train_loader)} batches | Dev = {len(dev_loader)} batches")

## 6. Huấn luyện Mô hình PhoBERT + CRF

In [ ]:
import numpy as np
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from seqeval.metrics import f1_score as seq_f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PhoBERT_CRF(model_name=MODEL_NAME, num_tags=len(LABEL2ID)).to(device)

# Thiết lập Optimizer với learning rate riêng biệt cho Transformer và CRF
optimizer_grouped_parameters = [
    {"params": [p for n, p in model.transformer.named_parameters()], "lr": LEARNING_RATE_TRANSFORMER},
    {"params": [p for n, p in model.classifier.named_parameters()], "lr": LEARNING_RATE_CRF},
    {"params": [p for n, p in model.crf.named_parameters()], "lr": LEARNING_RATE_CRF}
]
optimizer = AdamW(optimizer_grouped_parameters, weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_dev_f1 = 0.0
best_model_weights = None

print("BẮT ĐẦU HUẤN LUYỆN PHOBERT + CRF (Dự kiến ~5-7 phút)...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, (input_ids, attention_mask, first_subword_indices, word_labels, _) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss, _ = model(input_ids, attention_mask, first_subword_indices, word_labels)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        if (step + 1) % 100 == 0:
            print(f"  Epoch {epoch}/{NUM_EPOCHS} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_loss / len(train_loader)
    
    # Đánh giá trên tập Dev sau mỗi Epoch
    model.eval()
    all_gold_tags, all_pred_tags = [], []
    with torch.no_grad():
        for input_ids, attention_mask, first_subword_indices, word_labels, _ in dev_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                _, decoded_paths = model(input_ids, attention_mask, first_subword_indices)
            for b, pred_path in enumerate(decoded_paths):
                g_tags = [ID2LABEL[t] for t in word_labels[b]]
                p_tags = [ID2LABEL[t] for t in pred_path[:len(g_tags)]]
                all_gold_tags.append(g_tags)
                all_pred_tags.append(p_tags)
                
    dev_f1 = seq_f1_score(all_gold_tags, all_pred_tags)
    print(f"-> KẾT THÚC EPOCH {epoch}: Train Loss = {avg_train_loss:.4f} | Dev Token F1 = {dev_f1:.4f}")
    
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_model_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"   ★ Checkpoint mới tốt nhất! (F1: {best_dev_f1:.4f})")

print("\nHUẤN LUYỆN HOÀN TẤT!")
if best_model_weights is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_model_weights.items()})

## 7. Dự đoán & Đo lường Toàn diện 5 Trụ Cột trên Tập Dev

In [ ]:
model.eval()
all_dev_pred_tags = []

with torch.no_grad():
    for input_ids, attention_mask, first_subword_indices, word_labels, _ in dev_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            _, decoded_paths = model(input_ids, attention_mask, first_subword_indices)
            
        for b, pred_path in enumerate(decoded_paths):
            p_tags = [ID2LABEL[t] for t in pred_path]
            all_dev_pred_tags.append(p_tags)

pred_char_offsets_list = []
records = []

for sample, tags in zip(dev_samples, all_dev_pred_tags):
    while len(tags) < len(sample.tokens):
        tags.append("O")
    tags = tags[:len(sample.tokens)]
    
    offsets = bio_to_char_offsets(sample.tokens, tags, sample.content)
    pred_char_offsets_list.append(offsets)
    is_pred_toxic = 1 if ("B-T" in tags or len(offsets) > 0) else 0
    
    records.append({
        "sentence_id": sample.sentence_id,
        "content": sample.content,
        "gold_tags": " ".join(sample.tags),
        "pred_tags": " ".join(tags),
        "gold_char_offsets": str(sorted(list(sample.gold_char_offsets))),
        "pred_char_offsets": str(sorted(list(offsets))),
        "gold_toxic": sample.is_toxic,
        "pred_toxic": is_pred_toxic
    })

pred_filename = f"phobert_crf_{MODE}_dev_predictions.csv"
pred_df = pd.DataFrame(records)
pred_df.to_csv(pred_filename, index=False, encoding="utf-8")
print(f"-> ĐÃ XUẤT FILE PREDICTIONS: {pred_filename}")

## 8. Bảng Tổng Hợp Kết Quả Đánh Giá (5 Trụ Cột)

In [ ]:
def f1_per_char_span(gold, pred):
    if not gold and not pred: return 1.0, 1.0, 1.0
    if not gold or not pred: return 0.0, 0.0, 0.0
    ov = len(gold.intersection(pred))
    p, r = ov / len(pred), ov / len(gold)
    return p, r, (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

gold_char_offsets_list = [s.gold_char_offsets for s in dev_samples]
ps, rs, f1s = [], [], []
single_f1s, multi_f1s = [], []

for gold, pred in zip(gold_char_offsets_list, pred_char_offsets_list):
    p, r, f = f1_per_char_span(gold, pred)
    ps.append(p); rs.append(r); f1s.append(f)
    if gold:
        sorted_g = sorted(list(gold))
        nspans = 1 + sum(1 for i in range(1, len(sorted_g)) if sorted_g[i] != sorted_g[i-1] + 1)
        if nspans == 1: single_f1s.append(f)
        else: multi_f1s.append(f)

from sklearn.metrics import accuracy_score, f1_score as sk_f1
post_acc = accuracy_score([s.is_toxic for s in dev_samples], pred_df["pred_toxic"])
post_f1 = sk_f1([s.is_toxic for s in dev_samples], pred_df["pred_toxic"], average="binary")

exact, miss, spurious, under, over, shifted, clean_corr = 0, 0, 0, 0, 0, 0, 0
for gold, pred in zip(gold_char_offsets_list, pred_char_offsets_list):
    if not gold and not pred: clean_corr += 1
    elif gold and not pred: miss += 1
    elif not gold and pred: spurious += 1
    elif gold == pred: exact += 1
    else:
        inter = gold.intersection(pred)
        if len(inter) == 0: miss += 1
        elif pred.issubset(gold): under += 1
        elif gold.issubset(pred): over += 1
        elif min(gold) != min(pred) and max(gold) != max(pred): shifted += 1
        elif len(pred) < len(gold): under += 1
        else: over += 1

tot = len(dev_samples)
print("=" * 75)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH LAI: PHOBERT + LINEAR-CHAIN CRF")
print("=" * 75)
print(f"[Trụ cột 1 - Span & Plausibility]")
print(f"  • Character Macro-F1 (SemEval):  {np.mean(f1s):.4f}")
print(f"  • Char Precision / Recall:       {np.mean(ps):.4f} / {np.mean(rs):.4f}")
print(f"  • Char F1 Single-Span:           {np.mean(single_f1s):.4f}")
print(f"  • Char F1 Multiple-Span:         {np.mean(multi_f1s):.4f}")
print(f"\n[Trụ cột 2 - Post-level Classification]")
print(f"  • Post Accuracy:                 {post_acc:.4f}")
print(f"  • Post Binary-F1 (Toxic/Clean):  {post_f1:.4f}")
print(f"\n[Trụ cột 5 - Phân rã lỗi ranh giới (H3 Audit)]")
print(f"  • Exact Match:      {exact/tot*100:5.2f}%")
print(f"  • Under-span:       {under/tot*100:5.2f}%")
print(f"  • Over-span:        {over/tot*100:5.2f}%")
print(f"  • Shifted Boundary: {shifted/tot*100:5.2f}%")
print(f"  • Complete Miss:    {miss/tot*100:5.2f}%")
print(f"  • Spurious Span:    {spurious/tot*100:5.2f}%")
print(f"  • Clean Correct:    {clean_corr/tot*100:5.2f}%")
print("=" * 75)

## 9. Tải File Predictions về Máy Cá Nhân

In [ ]:
from google.colab import files
print(f"Đang tải {pred_filename} về máy của bạn...")
files.download(pred_filename)